In [1]:
!nvidia-smi

Sun Sep  6 12:54:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
!pip install -q -U \
    "transformers>=4.45,<5" \
    "peft>=0.12,<0.18" \
    "bitsandbytes>=0.43.3" \
    "trl>=0.10,<0.24" \
    "accelerate>=0.34" \
    "datasets>=2.20"

In [3]:
!pip install -q --force-reinstall --no-cache-dir "pyarrow==21.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 240.5 MB/s eta 0:00:00


In [4]:
import torch
import transformers
import peft
import bitsandbytes
import trl
import accelerate
import datasets

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.11.0+cu128
Transformers: 4.57.6
PEFT: 0.17.1
TRL: 0.23.1
Accelerate: 1.14.0
Datasets: 5.0.1

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [5]:
import json
from google.colab import drive

drive.mount('/content/drive')

data_dir = '/content/drive/MyDrive/pii-guardrail/data/processed'

with open(f'{data_dir}/train.json') as f:
    train_data = json.load(f)

with open(f'{data_dir}/test.json') as f:
    test_data = json.load(f)

print(f"Train examples: {len(train_data)}")
print(f"Test examples:  {len(test_data)}")
print(f"\nSample example:")
print(json.dumps(train_data[0], indent=2))

Mounted at /content/drive
Train examples: 1239
Test examples:  310

Sample example:
{
  "source": "Send alert to 7567890123.",
  "spans": [
    {
      "text": "7567890123",
      "category": "PHONE"
    }
  ],
  "origin": "handwritten_indian_phone"
}


In [6]:
def format_example(example):
    """
    Convert a dataset example into the instruction format Qwen expects.

    The model is instruction-tuned, meaning it expects:
    - A system message: what the model's job is
    - A user message: the input text
    - An assistant message: the expected output

    The output is a JSON array of identified PII spans.
    If no PII exists, the output is an empty array [].
    """

    source = example["source"]
    spans  = example["spans"]

    # Build the expected output — JSON string
    # This is what the model must learn to produce
    expected_output = json.dumps(spans, ensure_ascii=False)

    # System prompt — tells the model its exact job
    system_prompt = """You are a PII detection system. Given input text, identify all personally identifiable information and sensitive credentials.

Output ONLY a JSON array of detected spans. Each span has "text" (exact substring) and "category" (one of: PERSON, EMAIL, PHONE, AADHAAR, PAN, CREDIT_CARD, API_KEY, DB_CREDENTIAL).

If no PII or sensitive data is found, output exactly: []

Do not explain. Do not add any text before or after the JSON array."""

    # Format using Qwen's chat template structure
    # <|im_start|> and <|im_end|> are Qwen's special tokens
    formatted = (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        f"<|im_start|>user\n{source}<|im_end|>\n"
        f"<|im_start|>assistant\n{expected_output}<|im_end|>"
    )

    return {"text": formatted}

# Format all examples
train_formatted = [format_example(ex) for ex in train_data]
test_formatted  = [format_example(ex) for ex in test_data]

print(f"Train formatted: {len(train_formatted)}")
print(f"Test formatted:  {len(test_formatted)}")
print("\nSample formatted example:")
print(train_formatted[0]["text"])

Train formatted: 1239
Test formatted:  310

Sample formatted example:
<|im_start|>system
You are a PII detection system. Given input text, identify all personally identifiable information and sensitive credentials.

Output ONLY a JSON array of detected spans. Each span has "text" (exact substring) and "category" (one of: PERSON, EMAIL, PHONE, AADHAAR, PAN, CREDIT_CARD, API_KEY, DB_CREDENTIAL).

If no PII or sensitive data is found, output exactly: []

Do not explain. Do not add any text before or after the JSON array.<|im_end|>
<|im_start|>user
Send alert to 7567890123.<|im_end|>
<|im_start|>assistant
[{"text": "7567890123", "category": "PHONE"}]<|im_end|>


In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Pad token: {tokenizer.pad_token}")
print(f"EOS token: {tokenizer.eos_token}")

# Check token length distribution
# This tells us what MAX_SEQ_LENGTH to set
# Too short = examples get cut off, model learns wrong outputs
# Too long = wastes VRAM on padding

lengths = []
for ex in train_formatted:
    tokens = tokenizer(ex["text"], return_tensors="pt")
    lengths.append(tokens["input_ids"].shape[1])

import numpy as np
print(f"\nToken length stats:")
print(f"  Min:    {min(lengths)}")
print(f"  Max:    {max(lengths)}")
print(f"  Mean:   {np.mean(lengths):.0f}")
print(f"  P90:    {np.percentile(lengths, 90):.0f}")
print(f"  P95:    {np.percentile(lengths, 95):.0f}")
print(f"  P99:    {np.percentile(lengths, 99):.0f}")

# Count examples that would be cut at various limits
for limit in [256, 512, 768, 1024]:
    cut = sum(1 for l in lengths if l > limit)
    print(f"  Examples > {limit} tokens: {cut} ({100*cut//len(lengths)}%)")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: Qwen/Qwen2.5-1.5B-Instruct
Vocab size: 151643
Pad token: <|endoftext|>
EOS token: <|im_end|>

Token length stats:
  Min:    123
  Max:    356
  Mean:   181
  P90:    237
  P95:    252
  P99:    295
  Examples > 256 tokens: 56 (4%)
  Examples > 512 tokens: 0 (0%)
  Examples > 768 tokens: 0 (0%)
  Examples > 1024 tokens: 0 (0%)


In [8]:
#Cell 6 — Load model in 4-bit quantization:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 512

# 4-bit quantization config
# This is the Q in QLoRA
# Loads the frozen base model using only 4 bits per parameter
# instead of 16, reducing VRAM by ~4x for the base model weights
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",         # NormalFloat4: better than int4 for weights
    bnb_4bit_compute_dtype=torch.float16,  # compute in fp16 even though stored in 4bit
    bnb_4bit_use_double_quant=True,    # quantize the quantization constants too
)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model loaded")
print(f"Model dtype: {model.dtype}")

# Check VRAM used so far
print(f"VRAM used after model load: "
      f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM reserved: "
      f"{torch.cuda.memory_reserved() / 1e9:.2f} GB")

Loading model...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded
Model dtype: torch.float16
VRAM used after model load: 1.16 GB
VRAM reserved: 1.58 GB


In [9]:
!pip install -q -U "bitsandbytes>=0.46.1"

In [10]:

#Cell 7 — Apply LoRA adapters:

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training
# This does gradient checkpointing setup and
# ensures the frozen quantized layers work with trainable adapters
model = prepare_model_for_kbit_training(model)

# LoRA configuration
# r=16: rank of the adapter matrices
# Higher rank = more capacity to learn, more memory
# r=16 is standard starting point for a task this size
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,          # scaling factor: alpha/r = 2 is standard
    lora_dropout=0.05,      # small dropout on adapter weights
    bias="none",
    task_type="CAUSAL_LM",

    # Which layers to apply LoRA to
    # q_proj and v_proj are standard from original LoRA paper
    # Adding k_proj, o_proj, and gate/up/down proj gives more capacity
    # for a structured output task like JSON span detection
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

# Print trainable parameter count
# This is what you quote in interviews
model.print_trainable_parameters()

print(f"\nVRAM after LoRA: "
      f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

VRAM after LoRA: 1.70 GB


In [11]:
#Cell 8 — Prepare dataset for trainer:
from datasets import Dataset

# Convert to HuggingFace Dataset format
train_hf = Dataset.from_list(train_formatted)
test_hf  = Dataset.from_list(test_formatted)

print(f"Train dataset: {train_hf}")
print(f"Test dataset:  {test_hf}")

# Tokenize function
# We tokenize the full formatted text including the expected output
# The trainer will handle the loss masking internally
def tokenize(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,    # SFTTrainer handles padding per batch
    )
    result["labels"] = result["input_ids"].copy()
    return result

train_tokenized = train_hf.map(tokenize, remove_columns=["text"])
test_tokenized  = test_hf.map(tokenize,  remove_columns=["text"])

print(f"\nTrain tokenized: {train_tokenized}")
print(f"Test tokenized:  {test_tokenized}")

# Verify one example
print(f"\nSample input_ids length: {len(train_tokenized[0]['input_ids'])}")

Train dataset: Dataset({
    features: ['text'],
    num_rows: 1239
})
Test dataset:  Dataset({
    features: ['text'],
    num_rows: 310
})


Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

Map:   0%|          | 0/310 [00:00<?, ? examples/s]


Train tokenized: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1239
})
Test tokenized:  Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 310
})

Sample input_ids length: 154


In [19]:
from trl import SFTTrainer, SFTConfig
import torch

# Find the token IDs for the assistant header
# We will mask everything BEFORE this sequence in labels
RESPONSE_TEMPLATE = "<|im_start|>assistant\n"
response_token_ids = tokenizer.encode(
    RESPONSE_TEMPLATE,
    add_special_tokens=False
)
print(f"Response template token IDs: {response_token_ids}")
print(f"Response template decoded: {tokenizer.decode(response_token_ids)}")

def mask_labels(example):
    """
    Mask all tokens before the assistant response.
    Set labels = -100 for everything up to and including
    the assistant header tokens.
    Only the actual JSON output contributes to loss.
    """
    input_ids = example["input_ids"]
    labels    = list(input_ids).copy()

    # Find where the assistant response starts
    template_len = len(response_token_ids)
    found = False

    for i in range(len(input_ids) - template_len + 1):
        if input_ids[i:i+template_len] == response_token_ids:
            # Mask everything up to and including the template
            for j in range(i + template_len):
                labels[j] = -100
            found = True
            break

    if not found:
        # If template not found, mask everything (safety fallback)
        labels = [-100] * len(labels)

    example["labels"] = labels
    return example

# Apply masking to both datasets
# We need to work with the raw formatted text, not pre-tokenized

def tokenize_and_mask(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    input_ids = result["input_ids"]
    labels = input_ids.copy()

    template_len = len(response_token_ids)
    found = False

    for i in range(len(input_ids) - template_len + 1):
        if input_ids[i:i+template_len] == response_token_ids:
            for j in range(i + template_len):
                labels[j] = -100
            found = True
            break

    if not found:
        labels = [-100] * len(labels)

    result["labels"] = labels
    return result

# Apply to HF datasets
train_masked = train_hf.map(tokenize_and_mask, remove_columns=["text"])
test_masked  = test_hf.map(tokenize_and_mask,  remove_columns=["text"])

print(f"\nTrain masked: {len(train_masked)} examples")
print(f"Test masked:  {len(test_masked)} examples")

Response template token IDs: [151644, 77091, 198]
Response template decoded: <|im_start|>assistant



Map:   0%|          | 0/1239 [00:00<?, ? examples/s]

Map:   0%|          | 0/310 [00:00<?, ? examples/s]


Train masked: 1239 examples
Test masked:  310 examples


In [20]:
# Verify on one batch
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

# Quick manual check on first example
ex = train_masked[0]
input_ids = ex["input_ids"]
labels    = ex["labels"]

masked_count   = sum(1 for l in labels if l == -100)
unmasked_count = sum(1 for l in labels if l != -100)
total          = len(labels)

print(f"Total tokens:      {total}")
print(f"Masked (-100):     {masked_count} ({100*masked_count//total}%)")
print(f"In loss:           {unmasked_count} ({100*unmasked_count//total}%)")

# Show the boundary — last few masked and first few unmasked
print(f"\nBoundary between masked and unmasked:")
prev_masked = True
for i, (inp, lab) in enumerate(zip(input_ids, labels)):
    is_masked = (lab == -100)
    if prev_masked and not is_masked:
        # Show 3 before and 5 after the boundary
        start = max(0, i-3)
        end   = min(len(input_ids), i+5)
        for j in range(start, end):
            token  = tokenizer.decode([input_ids[j]])
            status = "MASKED" if labels[j] == -100 else "IN LOSS"
            print(f"  {repr(token):<25} {status}")
        break
    prev_masked = is_masked

Total tokens:      154
Masked (-100):     130 (84%)
In loss:           24 (15%)

Boundary between masked and unmasked:
  '<|im_start|>'            MASKED
  'assistant'               MASKED
  '\n'                      MASKED
  '['                       IN LOSS
  '{"'                      IN LOSS
  'text'                    IN LOSS
  '":'                      IN LOSS
  ' "'                      IN LOSS


In [22]:
from transformers import DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig
import torch

training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/pii-guardrail/models/qwen-pii",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=25,
    report_to="none",
    dataloader_num_workers=0,
    seed=42,
    dataset_text_field=None,
    max_length=MAX_SEQ_LENGTH,   # fixed: was max_seq_length
    packing=False,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_masked,
    eval_dataset=test_masked,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Trainer ready")
print(f"Steps per epoch: {len(trainer.get_train_dataloader())}")
print(f"Total steps: {int(training_args.num_train_epochs * len(trainer.get_train_dataloader()))}")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Truncating train dataset:   0%|          | 0/1239 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/310 [00:00<?, ? examples/s]

Trainer ready
Steps per epoch: 310
Total steps: 930
VRAM: 1.70 GB


In [23]:
batch = next(iter(trainer.get_train_dataloader()))
active = (batch["labels"] != -100).sum().item()
total  = batch["labels"].numel()
print(f"Tokens in loss: {active}/{total} ({100*active/total:.1f}%)")

Tokens in loss: 73/704 (10.4%)


In [ ]:
#Training

import os
os.makedirs(
    "/content/drive/MyDrive/pii-guardrail/models/qwen-pii",
    exist_ok=True
)

print("Starting training...")
train_result = trainer.train()
print(f"\nTraining complete")
print(f"Final training loss: {train_result.training_loss:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Step,Training Loss,Validation Loss
